# Experiment 49: Prediction Family Mining

This experiment studies the full external prediction library instead of treating every submission as an independent model. The goal is to identify prediction families, measure their diversity, find strong family representatives, and search constrained rank-based ensembles with Submission 14.

The external predictions are treated as candidate signals, not ground truth.

In [10]:
from pathlib import Path
import zipfile
import time
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.cluster import AgglomerativeClustering

ROOT = Path(r'C:\\Users\\aakif\\Documents\\DataCompetition')
EXT = ROOT / 'external_data' / 's6e9_zoom_zoom_baseline'
ARCHIVE = EXT / 'ranked_predictions_latest.zip.bin'
CATALOG = EXT / 'ranked_catalog_latest.csv'
S14_PATH = ROOT / 'submissions' / 'submission_14.csv'
OUT = ROOT / 'submissions' / 'experiment_49_prediction_family_mining'
OUT.mkdir(parents=True, exist_ok=True)

print('EXP49: PREDICTION FAMILY MINING')
print(f'Archive: {ARCHIVE}')
print(f'Catalog: {CATALOG}')
print(f'Submission 14: {S14_PATH}')

EXP49: PREDICTION FAMILY MINING
Archive: C:\Users\aakif\Documents\DataCompetition\external_data\s6e9_zoom_zoom_baseline\ranked_predictions_latest.zip.bin
Catalog: C:\Users\aakif\Documents\DataCompetition\external_data\s6e9_zoom_zoom_baseline\ranked_catalog_latest.csv
Submission 14: C:\Users\aakif\Documents\DataCompetition\submissions\submission_14.csv


In [11]:
catalog = pd.read_csv(CATALOG)
s14 = pd.read_csv(S14_PATH)

print('Catalog columns:')
print(catalog.columns.tolist())
print(f'Catalog rows: {len(catalog):,}')
print(f'Submission 14 rows: {len(s14):,}')
print()
print('First catalog rows:')
display(catalog.head(3))
        

Catalog columns:
['origin', 'author', 'submission_id', 'version_id', 'public_score', 'own_sort_position', 'source_url', 'sha256', 'csv_path', 'binding', 'display_order', 'displayed_score_rank', 'rows', 'prediction_kind']
Catalog rows: 649
Submission 14 rows: 286,571

First catalog rows:


,origin,author,submission_id,version_id,public_score,own_sort_position,source_url,sha256,csv_path,binding,display_order,displayed_score_rank,rows,prediction_kind
0,own,jazivxt,56510229,NaN,0.94657,1.0,https://www.kaggle.com/competitions/playground...,dc3c0ba4df8f02756dbe4adc7a717fd16e5a43fab4bc23...,own/own_001_56510229.csv,Exact scored submission ID,1,1,286571,probability
1,own,jazivxt,56509658,NaN,0.94657,2.0,https://www.kaggle.com/competitions/playground...,616572e1f73e76c2df8e861ba46a20a92b99db9aec0f12...,own/own_002_56509658.csv,Exact scored submission ID,2,1,286571,probability
2,own,jazivxt,56509671,NaN,0.94657,3.0,https://www.kaggle.com/competitions/playground...,e82277d75b6b8fcc7a144b17b89d422da2f3bae4e4fbf2...,own/own_003_56509671.csv,Exact scored submission ID,3,1,286571,probability


In [12]:
zf = zipfile.ZipFile(ARCHIVE, 'r')
members = [m for m in zf.namelist() if m.lower().endswith('.csv')]

print(f'Total ZIP members: {len(zf.namelist()):,}')
print(f'CSV members: {len(members):,}')
print()
print('First 20 CSV members:')
for m in members[:20]:
    print(m)


Total ZIP members: 651
CSV members: 650

First 20 CSV members:
own/own_001_56510229.csv
own/own_002_56509658.csv
own/own_003_56509671.csv
own/own_004_56509901.csv
own/own_005_56510108.csv
own/own_006_56477811.csv
own/own_007_56509863.csv
own/own_008_56477723.csv
own/own_009_56509923.csv
own/own_010_56452029.csv
own/own_011_56451994.csv
own/own_012_56451916.csv
own/own_013_56451979.csv
own/own_014_56369364.csv
own/own_015_56451924.csv
own/own_016_56451904.csv
own/own_017_56369287.csv
own/own_018_56451952.csv
own/own_019_56451936.csv
own/own_020_56451943.csv


In [13]:
# Robust archive mapping.
# The catalog and ZIP do not necessarily use identical path strings.
# We therefore inspect every CSV header and ID sequence and map by the
# catalog filename when possible, with deterministic fallback matching.

def normalize_name(x):
    x = str(x).replace('\\', '/').strip()
    x = x.lstrip('./')
    return x

member_by_name = {}
member_by_base = {}

for m in members:
    n = normalize_name(m)
    member_by_name[n] = m
    member_by_base[Path(n).name] = m

catalog_name_columns = [
    c for c in catalog.columns
    if any(k in c.lower() for k in ['file', 'path', 'name', 'filename'])
]

print('Possible catalog filename columns:', catalog_name_columns)

def catalog_name_candidates(row):
    vals = []
    for col in catalog_name_columns:
        value = row[col]
        if pd.notna(value):
            value = normalize_name(value)
            vals.append(value)
            vals.append(Path(value).name)
    return vals

mapping = []
used = set()

for i, row in catalog.iterrows():
    found = None

    for candidate in catalog_name_candidates(row):
        if candidate in member_by_name:
            found = member_by_name[candidate]
            break
        if candidate in member_by_base:
            found = member_by_base[candidate]
            break

    mapping.append(found)

catalog['archive_member'] = mapping

print(f'Filename/path matches: {catalog.archive_member.notna().sum():,}')
print(f'Unmatched: {catalog.archive_member.isna().sum():,}')


Possible catalog filename columns: ['csv_path']
Filename/path matches: 649
Unmatched: 0


In [14]:
# If filename mapping was incomplete, build a content-based map.
# We use the first few rows of each CSV to identify the common prediction format.

if catalog['archive_member'].isna().any():
    print('Building fallback content signatures...')

    unmatched_catalog = catalog[catalog.archive_member.isna()].index.tolist()
    available_members = [m for m in members if m not in set(catalog.archive_member.dropna())]

    print(f'Unmatched catalog rows: {len(unmatched_catalog):,}')
    print(f'Unused CSV members: {len(available_members):,}')

    # The archive contains 649 prediction CSVs plus one catalog CSV.
    # If names cannot be mapped, the catalog order and archive order are
    # compared through the prediction structure before accepting a mapping.
    if len(unmatched_catalog) == len(available_members):
        print('Attempting deterministic remaining-member mapping...')

        # Use sorted member names as a deterministic fallback only when
        # every unmatched catalog row has exactly one unused CSV member.
        remaining = sorted(available_members)
        for idx, member in zip(unmatched_catalog, remaining):
            catalog.loc[idx, 'archive_member'] = member

print()
print(f'FINAL MAPPED: {catalog.archive_member.notna().sum():,}/{len(catalog):,}')

if catalog.archive_member.isna().any():
    print('\nUNMATCHED CATALOG ROWS:')
    display(catalog[catalog.archive_member.isna()].head(20))
    raise RuntimeError('Archive mapping is incomplete. Do not continue.')

print('Archive mapping complete.')


FINAL MAPPED: 649/649
Archive mapping complete.


In [15]:
# Validate every mapped prediction before loading the full matrix.
print('Validating archive members...')

ids = s14['id'].to_numpy()
valid_rows = []

for k, row in catalog.iterrows():
    member = row.archive_member
    try:
        with zf.open(member) as f:
            head = pd.read_csv(f, nrows=5)

        if 'id' not in head.columns or 'Will_Buy_EV' not in head.columns:
            continue

        valid_rows.append(k)
    except Exception as e:
        print(f'Invalid member {member}: {e}')

catalog = catalog.iloc[valid_rows].reset_index(drop=True)
print(f'Validated predictions: {len(catalog):,}')

if len(catalog) < 600:
    raise RuntimeError(f'Only {len(catalog)} predictions validated. Expected approximately 649.')

Validating archive members...
Validated predictions: 649


In [ ]:
# Load the complete prediction matrix.
X = np.empty((len(ids), len(catalog)), dtype=np.float32)

start = time.time()

for j, row in catalog.iterrows():
    with zf.open(row.archive_member) as f:
        df = pd.read_csv(f)

    if len(df) != len(ids):
        raise ValueError(
            f'Row mismatch for {row.archive_member}: '
            f'{len(df)} vs {len(ids)}'
        )

    if not df['id'].equals(s14['id']):
        df = df.set_index('id').loc[s14['id']].reset_index()

    X[:, j] = df['Will_Buy_EV'].to_numpy(dtype=np.float32)

    if (j + 1) % 50 == 0:
        print(f'Loaded {j + 1}/{len(catalog)}')

print(f'Prediction matrix: {X.shape}')
print(f'Load time: {(time.time() - start) / 60:.1f} minutes')

Loaded 50/649
Loaded 100/649
Loaded 150/649
Loaded 200/649
Loaded 250/649
Loaded 300/649
Loaded 350/649
Loaded 400/649


In [ ]:
# Rank transform.
R = np.empty_like(X, dtype=np.float32)

for j in range(X.shape[1]):
    R[:, j] = rankdata(X[:, j], method='average').astype(np.float32)

s14_rank = rankdata(
    s14['Will_Buy_EV'].to_numpy(),
    method='average'
).astype(np.float32)

print(f'Rank matrix: {R.shape}')

In [ ]:
# Fixed sample for family discovery.
rng = np.random.default_rng(2026)
sample_n = min(50000, len(ids))
sample_idx = rng.choice(len(ids), size=sample_n, replace=False)

Rs = R[sample_idx].astype(np.float64)
Rs -= Rs.mean(axis=0)
Rs /= Rs.std(axis=0) + 1e-12

corr = (Rs.T @ Rs) / len(sample_idx)
corr = np.clip(corr, -1, 1)
dist = 1 - corr
np.fill_diagonal(dist, 0)

upper = np.triu_indices_from(corr, k=1)

print(f'Pairwise rank correlation matrix: {corr.shape}')
print(
    f'Off-diagonal Spearman range: '
    f'{corr[upper].min():.6f} -> {corr[upper].max():.6f}'
)


In [ ]:
# Cluster the prediction library into rank-similarity families.
n_clusters = min(30, max(10, len(catalog) // 20))

cluster_model = AgglomerativeClustering(
    n_clusters=n_clusters,
    metric='precomputed',
    linkage='average'
)

labels = cluster_model.fit_predict(dist)
catalog['family'] = labels

family_rows = []

for family in sorted(catalog.family.unique()):
    sub = catalog[catalog.family == family]
    best = sub.sort_values('public_score', ascending=False).iloc[0]

    family_rows.append({
        'family': int(family),
        'size': len(sub),
        'best_public_score': best.public_score,
        'best_author': best.get('author', ''),
        'best_submission_id': best.get('submission_id', ''),
        'best_archive_member': best.archive_member
    })

families = pd.DataFrame(family_rows).sort_values(
    ['best_public_score', 'size'],
    ascending=[False, False]
).reset_index(drop=True)

families.to_csv(OUT / 'exp49_families.csv', index=False)

print(f'Families found: {len(families)}')
display(families.head(30))

In [ ]:
# Select the strongest public-score representative from each family.
representatives = []

for family in sorted(catalog.family.unique()):
    sub = catalog[catalog.family == family].sort_values(
        'public_score', ascending=False
    )
    representatives.append(sub.iloc[0])

reps = pd.DataFrame(representatives).sort_values(
    'public_score', ascending=False
).reset_index(drop=True)

reps.to_csv(OUT / 'exp49_family_representatives.csv', index=False)

print(f'Family representatives: {len(reps)}')
display(
    reps[['family', 'public_score', 'author', 'submission_id']].head(30)
)

In [ ]:
# Measure full-rank diversity against Submission 14.
rows = []

for _, row in reps.iterrows():
    idx = catalog.index[
        catalog.archive_member == row.archive_member
    ][0]

    rho = spearmanr(s14_rank, R[:, idx]).statistic
    absdiff = np.abs(
        s14_rank.astype(np.float64) -
        R[:, idx].astype(np.float64)
    )

    rows.append({
        'family': row.family,
        'public_score': row.public_score,
        'author': row.get('author', ''),
        'submission_id': row.get('submission_id', ''),
        'spearman_s14': rho,
        'mean_rank_shift': absdiff.mean(),
        'median_rank_shift': np.median(absdiff),
        'p99_rank_shift': np.percentile(absdiff, 99),
        'archive_member': row.archive_member
    })

diversity = pd.DataFrame(rows).sort_values(
    ['public_score', 'mean_rank_shift'],
    ascending=[False, False]
).reset_index(drop=True)

diversity.to_csv(OUT / 'exp49_family_diversity.csv', index=False)

display(diversity.head(30))

In [ ]:
# Build a small family-consensus rank prediction.
top_n = min(12, len(diversity))
top = diversity.head(top_n)

indices = []

for _, row in top.iterrows():
    indices.append(
        catalog.index[
            catalog.archive_member == row.archive_member
        ][0]
    )

family_rank_mean = R[:, indices].mean(axis=1)

def rank01(x):
    return rankdata(x, method='average') / len(x)

s14_r01 = rank01(s14_rank)
family_r01 = rank01(family_rank_mean)

candidate_rows = []

for w in [0.01, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.30]:
    pred = (1 - w) * s14_r01 + w * family_r01
    shift = np.abs(rankdata(pred) - rankdata(s14_r01))

    candidate_rows.append({
        'candidate': f'family_consensus_{w:.3f}',
        'external_weight': w,
        'spearman_to_s14': spearmanr(s14_r01, pred).statistic,
        'mean_rank_shift': shift.mean(),
        'median_rank_shift': np.median(shift),
        'p99_rank_shift': np.percentile(shift, 99)
    })

candidate_summary = pd.DataFrame(candidate_rows)
candidate_summary.to_csv(OUT / 'exp49_candidate_summary.csv', index=False)

display(candidate_summary)

In [ ]:
# Score-weighted family consensus.
scores = reps['public_score'].to_numpy(dtype=float)
z = (scores - np.median(scores)) / (np.std(scores) + 1e-12)
weights = np.exp(np.clip(z, -2, 2))
weights /= weights.sum()

rep_indices = [
    catalog.index[
        catalog.archive_member == member
    ][0]
    for member in reps.archive_member
]

weighted_family_rank = np.average(
    R[:, rep_indices],
    axis=1,
    weights=weights
)

weighted_r01 = rank01(weighted_family_rank)
rows = []

for w in [0.01, 0.025, 0.05, 0.075, 0.10, 0.15, 0.20]:
    pred = (1 - w) * s14_r01 + w * weighted_r01
    shift = np.abs(rankdata(pred) - rankdata(s14_r01))

    rows.append({
        'candidate': f'score_weighted_family_{w:.3f}',
        'external_weight': w,
        'spearman_to_s14': spearmanr(s14_r01, pred).statistic,
        'mean_rank_shift': shift.mean(),
        'median_rank_shift': np.median(shift),
        'p99_rank_shift': np.percentile(shift, 99)
    })

score_candidates = pd.DataFrame(rows)
score_candidates.to_csv(
    OUT / 'exp49_score_weighted_candidates.csv',
    index=False
)

display(score_candidates)

In [ ]:
print('EXP49 COMPLETE')
print(f'Output directory: {OUT}')